# PySpark Assignment:
This notebook is a step-by-step PySpark assignment using the `sample-superstore.csv` dataset. It covers Spark session creation, data loading, cleaning, filtering, aggregation, schema modification, and a final processing pipeline.

In [1]:
# Install PySpark if it is not already installed in the notebook environment.
# If PySpark is already installed, this cell will simply confirm it.
!pip install pyspark --quiet

## 1. Create a Spark Session
A Spark Session is the entry point to using Spark with DataFrames and SQL.

In [2]:
from pyspark.sql import SparkSession
import os

# Create a Spark Session for this notebook.
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

spark = SparkSession.builder \
    .appName("Superstore Sales Analysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

print("Spark session created successfully")
print("Spark version:", spark.version)

Spark session created successfully
Spark version: 4.0.2


## 2. Load the Dataset
We load the `sample-superstore.csv` file as a Spark DataFrame with a header and automatic schema inference.

In [4]:
# Dataset path - update this file name if needed for your workspace.
file_path = "Sample - Superstore.csv"

# Load the CSV file into a Spark DataFrame.
df = spark.read.csv(file_path, header=True, inferSchema=True)

print("Dataset loaded successfully")
print("Number of rows:", df.count())

Dataset loaded successfully
Number of rows: 9994


## 3. Display Schema and Sample Records
This helps us understand the data structure and verify the first few rows.

In [5]:
# Show the schema of the DataFrame.
df.printSchema()

# Show sample records from the DataFrame.
df.show(5, truncate=False)

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+------

## 4. Explain Spark DataFrame Immutability
Spark DataFrames are immutable, which means operations create new DataFrames instead of changing the original one.

In [6]:
from pyspark.sql.functions import col

# Create a new DataFrame with only a subset of columns.
selected_df = df.select("Order ID", "Product Name", "Sales")

print("Original DataFrame columns:", df.columns)
print("New selected DataFrame columns:", selected_df.columns)

# Confirm the original DataFrame is unchanged.
print("Original row count:", df.count())
print("Selected row count:", selected_df.count())

Original DataFrame columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
New selected DataFrame columns: ['Order ID', 'Product Name', 'Sales']
Original row count: 9994
Selected row count: 9994


## 5. Data Cleaning
We will check for missing values, handle them, remove duplicates, and handle empty values.

In [7]:
# Check for null values in each column.
null_counts = {column: df.filter(col(column).isNull()).count() for column in df.columns}
print("Null counts per column:")
for column, count in null_counts.items():
    print(f" - {column}: {count}")

Null counts per column:
 - Row ID: 0
 - Order ID: 0
 - Order Date: 0
 - Ship Date: 0
 - Ship Mode: 0
 - Customer ID: 0
 - Customer Name: 0
 - Segment: 0
 - Country: 0
 - City: 0
 - State: 0
 - Postal Code: 0
 - Region: 0
 - Product ID: 0
 - Category: 0
 - Sub-Category: 0
 - Product Name: 0
 - Sales: 0
 - Quantity: 0
 - Discount: 0
 - Profit: 0


In [9]:
# Remove rows where Customer Name is empty
from pyspark.sql.functions import col

clean_df = clean_df.filter(
    col("Customer Name") != ""
)
clean_df.show(5)

+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID| Customer Name|    Segment|      Country|         City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|   222|CA-2015-169397|12/24/2015|12/27/2015|   First Class|   JB-15925|Joni Blumstein|   Consumer|United States|       Dublin|          Ohio|      43017|   East|OFF-BI-10002852|Office Supplies|     Binder

In [10]:
# Replace empty values in Country column
from pyspark.sql.functions import col, when

clean_df = clean_df.withColumn(
    "Country",
    when(col("Country") == "", "Unknown")
    .otherwise(col("Country"))
)
print("Empty values handled successfully.")
clean_df.show(5)

Empty values handled successfully.
+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID| Customer Name|    Segment|      Country|         City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+--------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|   222|CA-2015-169397|12/24/2015|12/27/2015|   First Class|   JB-15925|Joni Blumstein|   Consumer|United States|       Dublin|          Ohio|      43017|   East|OFF-BI-1

## 6. Filtering Operations
We filter records by Sales, Category, and Region.

In [39]:
import pyspark.sql.functions as F

# Sales column to double using TRY_CAST and filter records
high_sales_df = clean_df.withColumn(
    "Sales_Double",
    F.expr("TRY_CAST(Sales AS DOUBLE)")
)
high_sales_df = high_sales_df.filter(
    F.col("Sales_Double") > 500
)
print("Records with Sales > 500:")
high_sales_df.show(5)

Records with Sales > 500:
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|         State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|Sales_Double|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+------------+
|  2929|US-2017-120390|10/19/2017|10/26/2017|Standard Class|   TH-21550|    Tracy Hopkins|Home Office|United States|   Burlingto

In [18]:
# Filter records for the "Technology" category.
technology_df = clean_df.filter(col("Category") == "Technology")
print("Technology category record count:", technology_df.count())
technology_df.show(5, truncate=False)

Technology category record count: 1847
+------+--------------+----------+----------+--------------+-----------+-----------------+--------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+-------------------------------------------------------------+-------+--------+--------+---------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name    |Segment |Country      |City         |State     |Postal Code|Region |Product ID     |Category  |Sub-Category|Product Name                                                 |Sales  |Quantity|Discount|Profit   |
+------+--------------+----------+----------+--------------+-----------+-----------------+--------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+-------------------------------------------------------------+-------+--------+--------+---------+
|391   |CA-2017-101798|12-11-2017|12/15/2017|Standard Class|MV-18190  

In [20]:
# Filter records for the "West" region.
west_region_df = clean_df.filter(col("Region") == "West")
print("West region record count:", west_region_df.count())
west_region_df.show(5, truncate=False)

West region record count: 3203
+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------------------------+-------+--------+--------+------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name    |Segment  |Country      |City         |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                          |Sales  |Quantity|Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------------------------+-------+--------+--------+------+
|566   |CA-2017-137099|12-07-2017|12-10-2017|First Class   |FP-14320   |Frank Preis      |Consumer |United States|Los Angeles  |California|90

## 7. Aggregation Operations
We use count, sum, avg, min, and max to analyze numeric data.

In [22]:
from pyspark.sql.functions import count, sum as spark_sum, avg, min as spark_min, max as spark_max, expr

#aggregations on the Sales column.
aggregation_df = clean_df.withColumn("Sales_Double", expr("TRY_CAST(Sales AS DOUBLE)")).agg(
    count("Sales_Double").alias("SalesCount"),
    spark_sum("Sales_Double").alias("SalesSum"),
    avg("Sales_Double").alias("SalesAvg"),
    spark_min("Sales_Double").alias("SalesMin"),
    spark_max("Sales_Double").alias("SalesMax")
)
aggregation_df.show(truncate=False)

+----------+------------------+------------------+--------+--------+
|SalesCount|SalesSum          |SalesAvg          |SalesMin|SalesMax|
+----------+------------------+------------------+--------+--------+
|9694      |2272449.8562999824|234.41818199917293|0.444   |22638.48|
+----------+------------------+------------------+--------+--------+



## 8. GroupBy Operations
GroupBy is a wide transformation in Spark and typically requires a shuffle of data across the cluster.

In [24]:
# Total Sales by Category.
category_sales_df = clean_df.withColumn("Sales_Double", expr("TRY_CAST(Sales AS DOUBLE)")).groupBy("Category").agg(spark_sum("Sales_Double").alias("TotalSales"))
print("Total Sales by Category")
category_sales_df.show(truncate=False)

Total Sales by Category
+---------------+-----------------+
|Category       |TotalSales       |
+---------------+-----------------+
|Office Supplies|703502.9280000012|
|Furniture      |733046.8613000009|
|Technology     |835900.067       |
+---------------+-----------------+



In [25]:
# Total Sales by Region.
region_sales_df = clean_df.withColumn("Sales_Double", expr("TRY_CAST(Sales AS DOUBLE)")).groupBy("Region").agg(spark_sum("Sales_Double").alias("TotalSales"))
print("Total Sales by Region")
region_sales_df.show(truncate=False)

Total Sales by Region
+-------+------------------+
|Region |TotalSales        |
+-------+------------------+
|South  |388983.5850000003 |
|Central|497800.87279999966|
|East   |672194.0540000005 |
|West   |713471.344500001  |
+-------+------------------+



In [26]:
# Average Sales by Category.
avg_category_sales_df = clean_df.withColumn("Sales_Double", expr("TRY_CAST(Sales AS DOUBLE)")).groupBy("Category").agg(avg("Sales_Double").alias("AvgSales"))
print("Average Sales by Category")
avg_category_sales_df.show(truncate=False)

Average Sales by Category
+---------------+------------------+
|Category       |AvgSales          |
+---------------+------------------+
|Office Supplies|121.69225531914915|
|Furniture      |353.44593119575745|
|Technology     |454.54054758020663|
+---------------+------------------+



## 9. Conditional Aggregation
We use filters on aggregated results to find categories above a sales threshold.

In [29]:
# Find categories with total sales above 100,000.
threshold = 100000
high_category_sales_df = category_sales_df.filter(col("TotalSales") > threshold)
print(f"Categories with Total Sales above {threshold}")
high_category_sales_df.show(truncate=False)

Categories with Total Sales above 100000
+---------------+-----------------+
|Category       |TotalSales       |
+---------------+-----------------+
|Office Supplies|703502.9280000012|
|Furniture      |733046.8613000009|
|Technology     |835900.067       |
+---------------+-----------------+



## 10. Schema Modifications
We rename a column and cast a column to a different data type.

In [37]:
# Rename Sales column
modified_df = clean_df.withColumnRenamed(
    "Sales",
    "TotalSales"
)

# Convert Quantity column to integer
modified_df = modified_df.withColumn(
    "Quantity",
    col("Quantity").cast("integer")
)

# Display updated schema
print("Schema after modifications")
modified_df.printSchema()

# Display sample records
modified_df.show(5)

Schema after modifications
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = false)
 |-- Order Date: string (nullable = false)
 |-- Ship Date: string (nullable = false)
 |-- Ship Mode: string (nullable = false)
 |-- Customer ID: string (nullable = false)
 |-- Customer Name: string (nullable = false)
 |-- Segment: string (nullable = false)
 |-- Country: string (nullable = false)
 |-- City: string (nullable = false)
 |-- State: string (nullable = false)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = false)
 |-- Product ID: string (nullable = false)
 |-- Category: string (nullable = false)
 |-- Sub-Category: string (nullable = false)
 |-- Product Name: string (nullable = false)
 |-- TotalSales: string (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- Discount: string (nullable = false)
 |-- Profit: double (nullable = false)

+------+--------------+----------+----------+--------------+-----------+--------------+-------

## 11. Wide Transformations and Shuffle Operations
`groupBy()` is an example of a wide transformation. Spark redistributes data across partitions to perform the aggregation.

In [32]:
# Demonstrate a wide transformation by grouping and counting records by Region and Category.
wide_df = clean_df.groupBy("Region", "Category").agg(count("Order ID").alias("OrderCount"))
print("Wide transformation result with grouped counts")
wide_df.show(truncate=False)

print("\nExplanation: groupBy() is a wide transformation, and Spark may shuffle data between partitions to compute these grouped results.")

Wide transformation result with grouped counts
+-------+---------------+----------+
|Region |Category       |OrderCount|
+-------+---------------+----------+
|South  |Office Supplies|995       |
|Central|Office Supplies|1422      |
|West   |Office Supplies|1897      |
|South  |Furniture      |332       |
|Central|Technology     |420       |
|Central|Furniture      |481       |
|South  |Technology     |293       |
|West   |Technology     |599       |
|West   |Furniture      |707       |
|East   |Office Supplies|1712      |
|East   |Furniture      |601       |
|East   |Technology     |535       |
+-------+---------------+----------+


Explanation: groupBy() is a wide transformation, and Spark may shuffle data between partitions to compute these grouped results.


## 12. Final Processing Pipeline
This pipeline removes duplicates, handles nulls, applies filtering, and produces a summarized result.

In [36]:
from pyspark.sql.functions import sum, avg, count, col, expr

# Step 1: Remove duplicate records
pipeline_df = clean_df.dropDuplicates()

# Step 2: Fill null values for Postal Code
pipeline_df = pipeline_df.fillna({
    "Postal Code": 0
})

# Step 3: Cast 'Sales' to Double and then filter records where Sales is greater than 100
pipeline_df = pipeline_df.withColumn("Sales_Double", expr("TRY_CAST(Sales AS DOUBLE)")) \
                         .filter(col("Sales_Double") > 100)

# Step 4: Group data by Region and aggregate using the new 'Sales_Double' column
final_summary_df = pipeline_df.groupBy("Region").agg(
    sum("Sales_Double").alias("Total_Sales"),
    avg("Sales_Double").alias("Average_Sales"),
    count("*").alias("Order_Count")
)

# Step 5: Display final result
print("Final Summary Report")
final_summary_df.show()

Final Summary Report
+-------+------------------+-----------------+-----------+
| Region|       Total_Sales|    Average_Sales|Order_Count|
+-------+------------------+-----------------+-----------+
|  South|358111.87900000025|596.8531316666671|        600|
|Central| 456668.4814000001|553.5375532121213|        825|
|   East| 617441.9049999992|588.6004814108668|       1049|
|   West| 651280.0084999993|530.7905529747345|       1227|
+-------+------------------+-----------------+-----------+



## 13. Insights and Conclusion
- Spark DataFrames are immutable: every operation returns a new DataFrame.
- Data cleaning is important for nulls, duplicates, and empty values.
- Filtering helps narrow down the rows for analysis.
- Aggregations and groupBy() let us summarize sales by category and region.
- Schema modifications can help make column names clearer and ensure correct numeric types.

This notebook demonstrates a complete Spark workflow from data loading to a final summarized result.